# DINOv3 vs EUPE — STM 缺陷语义分割

**放弃 DetectionHead**：per-token 全局 top-k 匹配对小目标检测效果很差。
**改用分割**：rectangle 标注渲染为 mask，用 DiceCELoss 做 3 类分割（bg/dark/bright）。

| class_id | 类别 | 标注形状 |
|----------|------|----------|
| 0 | background | — |
| 1 | dark_defect | rectangle |
| 2 | bright_defect | rectangle |

| Encoder | embed_dim | depth | 训练模式 |
|---------|-----------|-------|----------|
| DINOv3 ViT-L | 1024 | 24 | head_only |
| EUPE ViT-B | 768 | 12 | head_only |

**标注源**: `png-defect/*.json` → rectangle → mask
**Head**: DINOv3LinearSegmentationHead
**Loss**: DiceCELoss (CE + 0.5×Dice)

In [ ]:
from __future__ import annotations

import json, random, sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as nn_functional
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image as PILImage, ImageDraw

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for parent in [start, *start.parents]:
        if (parent / 'src' / 'lumen').exists():
            return parent
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DEF_DIR = REPO_ROOT / 'data' / 'stm_dataset' / 'FeTe-sxm' / 'png-defect'
DINOV3_PATH = REPO_ROOT / 'checkpoints' / 'dinov3-vitl16-pretrain-lvd1689m'
EUPE_PATH   = REPO_ROOT / 'checkpoints' / 'EUPE-ViT-B.pt'

CLASS_NAMES = ['background', 'dark_defect', 'bright_defect']
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_ID = {'dark_defect': 1, 'bright_defect': 2}

IMAGE_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 80
HEAD_LR = 5e-4
DICE_WEIGHT = 0.5
WEIGHT_DECAY = 1e-4
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
print(f'DEF_DIR: {DEF_DIR.exists()}')
print(f'DINOv3: {DINOV3_PATH.exists()}, EUPE: {EUPE_PATH.exists()}')

## 1. 缺陷分割数据集：rectangle → mask

In [ ]:
class DefectSegDataset(Dataset):
    """从 png-defect LabelMe JSON 构造语义分割数据集。rectangle 渲染为 mask。"""

    def __init__(self, data_dir, label_to_id, image_size=512, augment=False):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.label_to_id = label_to_id
        self.image_size = image_size
        self.augment = augment

        self.samples = []
        for jf in sorted(self.data_dir.glob('*.json')):
            png_path = jf.with_suffix('.png')
            if not png_path.exists():
                continue
            with open(jf, encoding='utf-8') as f:
                ann = json.load(f)
            shapes = []
            for s in ann.get('shapes', []):
                cid = self.label_to_id.get(s['label'])
                if cid is None:
                    continue
                pts = s['points']
                if not pts:
                    continue
                shapes.append({'label_id': cid, 'points': pts, 'shape_type': s.get('shape_type', 'polygon')})
            if not shapes:
                continue
            self.samples.append({
                'stem': jf.stem, 'png_path': png_path,
                'shapes': shapes,
                'img_h': ann.get('imageHeight', 0), 'img_w': ann.get('imageWidth', 0),
            })
        self.stems = [s['stem'] for s in self.samples]

    def __len__(self):
        return len(self.samples)

    def _render_mask(self, shapes, H, W):
        mask = PILImage.new('L', (W, H), color=0)
        drawer = ImageDraw.Draw(mask)
        for s in shapes:
            pts, cid = s['points'], s['label_id']
            st = s['shape_type']
            if st == 'polygon' and len(pts) >= 3:
                drawer.polygon([(float(p[0]), float(p[1])) for p in pts], fill=cid)
            elif st == 'rectangle' and len(pts) >= 2:
                x0, y0 = float(pts[0][0]), float(pts[0][1])
                x1, y1 = float(pts[1][0]), float(pts[1][1])
                drawer.rectangle([min(x0,x1), min(y0,y1), max(x0,x1), max(y0,y1)], fill=cid)
        return np.asarray(mask, dtype=np.int64)

    def _load_and_resize(self, sample):
        img = PILImage.open(sample['png_path']).convert('L')
        W, H = img.size
        mask = self._render_mask(sample['shapes'], H, W)
        S = self.image_size
        img_arr = np.asarray(img.resize((S, S), PILImage.BILINEAR), dtype=np.float32) / 255.0
        mask_pil = PILImage.fromarray(mask.astype(np.int32), mode='I')
        mask_arr = np.asarray(mask_pil.resize((S, S), PILImage.NEAREST), dtype=np.int64)
        return img_arr, mask_arr

    def _maybe_augment(self, image, mask):
        if not self.augment:
            return image, mask
        if random.random() < 0.5:
            image = image[:, ::-1].copy(); mask = mask[:, ::-1].copy()
        if random.random() < 0.5:
            image = image[::-1, :].copy(); mask = mask[::-1, :].copy()
        k = random.randint(0, 3)
        if k:
            image = np.rot90(image, k).copy(); mask = np.rot90(mask, k).copy()
        if random.random() < 0.5:
            image = np.clip(image * (1.0 + (random.random()-0.5)*0.4) + (random.random()-0.5)*0.2, 0.0, 1.0)
        return image, mask

    def __getitem__(self, index):
        s = self.samples[index]
        image, mask = self._load_and_resize(s)
        image, mask = self._maybe_augment(image, mask)
        return {'image': torch.from_numpy(image.astype(np.float32)).unsqueeze(0),
                'mask': torch.from_numpy(mask.astype(np.int64)), 'stem': s['stem']}


dataset = DefectSegDataset(DEF_DIR, LABEL_TO_ID, IMAGE_SIZE, augment=False)
print(f'Classes: {CLASS_NAMES}')
print(f'Loaded {len(dataset)} labeled samples')
for s in dataset.samples:
    ids = {}
    for sh in s['shapes']:
        name = CLASS_NAMES[sh['label_id']]
        ids[name] = ids.get(name, 0) + 1
    print(f'  {s["stem"]}: {len(s["shapes"])} shapes, labels={ids}')

## 2. 标注对齐验证

In [ ]:
def mask_to_rgb(mask):
    rgb = np.zeros((*mask.shape, 3))
    rgb[mask == 1] = [1, 0, 0]   # dark_defect: red
    rgb[mask == 2] = [1, 1, 0]   # bright_defect: yellow
    return rgb

n_show = len(dataset)
fig, axes = plt.subplots(2, n_show, figsize=(5*n_show, 10))
if n_show == 1:
    axes = axes[:, np.newaxis]
for i in range(n_show):
    sample = dataset[i]
    img = sample['image'][0].numpy()
    mask = sample['mask'].numpy()
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f"{sample['stem']}\nOriginal")
    axes[0, i].axis('off')
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].imshow(mask_to_rgb(mask), alpha=0.5)
    axes[1, i].set_title(f'Mask: dark={int((mask==1).sum())} bright={int((mask==2).sum())}')
    axes[1, i].axis('off')
legend = [Line2D([0],[0], color='red', linewidth=4, label='dark'),
          Line2D([0],[0], color='yellow', linewidth=4, label='bright')]
fig.legend(handles=legend, loc='upper center', ncol=2)
plt.tight_layout(); plt.show()

## 3. 训练/验证划分 + class 权重

In [ ]:
all_indices = list(range(len(dataset)))
random.Random(SEED).shuffle(all_indices)
n_val = max(1, len(all_indices) // 4)
val_indices = sorted(all_indices[:n_val])
train_indices = sorted(all_indices[n_val:])

val_dataset = Subset(dataset, val_indices)
print(f'Train: {len(train_indices)}, Val: {len(val_indices)}')
print(f'Train stems: {[dataset.stems[i] for i in train_indices]}')
print(f'Val stems:   {[dataset.stems[i] for i in val_indices]}')

dataset_aug = DefectSegDataset(DEF_DIR, LABEL_TO_ID, IMAGE_SIZE, augment=True)
train_dataset = Subset(dataset_aug, train_indices)

class_pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in train_indices:
    for cid in range(NUM_CLASSES):
        class_pixel_counts[cid] += int((dataset[i]['mask'].numpy() == cid).sum())
freqs = class_pixel_counts / class_pixel_counts.sum()
inv = 1.0 / np.clip(freqs, 1e-6, None)
class_weights = torch.tensor(inv / inv.sum() * NUM_CLASSES, dtype=torch.float32).to(DEVICE)

for cid, name in enumerate(CLASS_NAMES):
    print(f'  {cid} ({name}): {class_pixel_counts[cid]} px, weight={class_weights[cid].item():.4f}')

## 4. 构建两个 SegmentationTrainer + DiceCELoss

In [ ]:
from lumen.models import DINOv3Encoder
from lumen.models.eupe import EUPEEncoder
from lumen.training.downstream import SegmentationTrainer

class DiceCELoss(nn.Module):
    def __init__(self, ce_weight=None, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight, ignore_index=-1)
        self.dice_weight = dice_weight; self.smooth = smooth
    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        C = logits.shape[1]
        targets_oh = nn_functional.one_hot(targets, C).permute(0, 3, 1, 2).float()
        dice = 0.0; n = 0
        for c in range(1, C):
            inter = (probs[:, c] * targets_oh[:, c]).sum()
            union = probs[:, c].sum() + targets_oh[:, c].sum()
            if union > 0:
                dice += 1.0 - (2.0*inter + self.smooth) / (union + self.smooth); n += 1
        return ce_loss + self.dice_weight * dice / max(n, 1)

cweight = class_weights.detach().cpu()

# DINOv3
print('Loading DINOv3...')
dino_encoder = DINOv3Encoder(model_dir=DINOV3_PATH, device=DEVICE, local_files_only=True)
dino_trainer = SegmentationTrainer(encoder=dino_encoder, num_classes=NUM_CLASSES,
    trainability='head_only', segmentation_head_name='dinov3-linear',
    segmentation_loss='ce', scheduler_name='cosine', scheduler_t_max=EPOCHS,
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY).to(DEVICE)
dino_trainer.criterion = DiceCELoss(ce_weight=cweight, dice_weight=DICE_WEIGHT).to(DEVICE)
print(f'  DINOv3: embed_dim={dino_encoder.embed_dim}, trainable={sum(p.numel() for p in dino_trainer.parameters() if p.requires_grad):,}')

# EUPE
print('Loading EUPE...')
eupe_encoder = EUPEEncoder.from_pretrained(checkpoint_path=EUPE_PATH, device=DEVICE, strict=False)
eupe_encoder.auto_convert_input_channels = True
eupe_trainer = SegmentationTrainer(encoder=eupe_encoder, num_classes=NUM_CLASSES,
    trainability='head_only', segmentation_head_name='dinov3-linear',
    segmentation_loss='ce', scheduler_name='cosine', scheduler_t_max=EPOCHS,
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY).to(DEVICE)
eupe_trainer.criterion = DiceCELoss(ce_weight=cweight, dice_weight=DICE_WEIGHT).to(DEVICE)
print(f'  EUPE: embed_dim={eupe_encoder.embed_dim}, trainable={sum(p.numel() for p in eupe_trainer.parameters() if p.requires_grad):,}')
print(f'\nLoss: DiceCELoss(dice_weight={DICE_WEIGHT}), Both: head_only lr={HEAD_LR}')

## 5. 训练循环

In [ ]:
def collate(batch):
    return {'image': torch.stack([b['image'] for b in batch]),
            'mask': torch.stack([b['mask'] for b in batch]),
            'stem': [b['stem'] for b in batch]}

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False, collate_fn=collate)

@torch.no_grad()
def calc_miou(logits, mask):
    pred = logits.argmax(1)
    ious = []
    for c in range(NUM_CLASSES):
        p = pred==c; g = mask==c
        inter = (p & g).sum().item(); union = (p | g).sum().item()
        ious.append(inter/union if union>0 else float('nan'))
    valid = [v for v in ious if not np.isnan(v)]
    return {'miou': np.mean(valid) if valid else 0.0, 'ious': ious}

def train_one(name, trainer):
    history = {'train_loss':[], 'val_loss':[], 'val_miou':[]}
    best_miou = -1.0; best_epoch = 0
    CKPT_DIR = REPO_ROOT / 'artifacts' / 'defect_seg_compare'
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    BEST = CKPT_DIR / f'best_{name}.pt'
    print(f'\nTraining: {name}')
    for epoch in range(EPOCHS):
        trainer.train()
        losses = []
        for b in train_loader:
            b = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k,v in b.items()}
            losses.append(trainer.train_step(b)['loss'])
        t_loss = float(np.mean(losses))
        trainer.eval()
        v_losses, v_mious = [], []
        with torch.no_grad():
            for b in val_loader:
                b = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k,v in b.items()}
                logits = trainer(b['image'])
                v_losses.append(trainer.compute_loss(logits, b['mask']).item())
                v_mious.append(calc_miou(logits, b['mask'])['miou'])
        v_loss = float(np.mean(v_losses)); v_miou = float(np.mean(v_mious))
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['val_miou'].append(v_miou)
        if v_miou > best_miou:
            best_miou = v_miou; best_epoch = epoch
            torch.save({'head': trainer.head.state_dict(), 'epoch': epoch, 'miou': v_miou}, BEST)
        if (epoch+1) % 20 == 0 or epoch < 5:
            m = ' [BEST]' if v_miou >= best_miou else ''
            print(f'  [{name}] {epoch+1:3d}/{EPOCHS}: train={t_loss:.4f} val={v_loss:.4f} mIoU={v_miou:.4f}{m}')
    print(f'\nDone. Best mIoU={best_miou:.4f} (epoch {best_epoch})')
    return history, best_miou, best_epoch, BEST

dino_h, dino_miou, dino_be, DINO_BEST = train_one('DINOv3_ViT-L', dino_trainer)
eupe_h, eupe_miou, eupe_be, EUPE_BEST = train_one('EUPE_ViT-B', eupe_trainer)
print(f'\nDINOv3: {dino_miou:.4f} | EUPE: {eupe_miou:.4f} | Best: {"EUPE" if eupe_miou>dino_miou else "DINOv3"}')

## 6. 训练曲线 + 预测对比 + NMS 后处理

In [ ]:
# ── 曲线 ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for h, c, n in [(dino_h,'C0','DINOv3'), (eupe_h,'C1','EUPE')]:
    ax1.plot(h['train_loss'], c+'-', alpha=0.4, label=f'{n} train')
    ax1.plot(h['val_loss'], c+'-', linewidth=2, label=f'{n} val')
    ax2.plot(h['val_miou'], c+'-', linewidth=2, label=f'{n} best={max(h["val_miou"]):.4f}')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('DiceCELoss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU'); ax2.set_title('Validation mIoU'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ── 加载 best ──
dino_trainer.head.load_state_dict(torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)['head'])
dino_trainer.eval()
eupe_trainer.head.load_state_dict(torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)['head'])
eupe_trainer.eval()

# ── 把分割 mask 转换为 bbox（后处理，方便对比原始标注） ──
from scipy import ndimage

def mask_to_bboxes(mask, min_area=4):
    """从分割 mask 提取连通区域 → bbox (cx,cy,w,h) 归一化。"""
    H, W = mask.shape
    boxes = []
    for cid in [1, 2]:
        labeled, n = ndimage.label(mask == cid)
        for i in range(1, n+1):
            ys, xs = np.where(labeled == i)
            if len(ys) < min_area:
                continue
            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()
            cx = (x1+x2)/2/W; cy = (y1+y2)/2/H
            w = (x2-x1+1)/W; h = (y2-y1+1)/H
            boxes.append((cid, cx, cy, w, h))
    return boxes

# ── 预测可视化 ──
@torch.no_grad()
def infer(trainer, img_t):
    return trainer(img_t).argmax(1)[0].cpu().numpy()

n_show = len(val_indices)
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4*n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)
for row, idx in enumerate(val_indices):
    s = dataset[idx]; img_np = s['image'][0].numpy()
    img_t = s['image'].unsqueeze(0).to(DEVICE); gt = s['mask'].numpy()
    d_pred = infer(dino_trainer, img_t); e_pred = infer(eupe_trainer, img_t)
    
    titles = ['Input', 'Ground Truth', 'DINOv3', 'EUPE']
    masks = [None, gt, d_pred, e_pred]
    for c, (title, mask) in enumerate(zip(titles, masks)):
        axes[row, c].imshow(img_np, cmap='gray')
        if mask is not None:
            axes[row, c].imshow(mask_to_rgb(mask), alpha=0.5)
            dark = int((mask==1).sum()); bright = int((mask==2).sum())
            axes[row, c].set_title(f'{title}\ndark={dark} bright={bright}', fontsize=8)
        else:
            axes[row, c].set_title(f"{s['stem']}", fontsize=8)
        axes[row, c].axis('off')
winner = 'EUPE' if eupe_miou > dino_miou else 'DINOv3'
fig.suptitle(f'Defect Segmentation Validation — {winner} wins ({max(dino_miou, eupe_miou):.4f})', fontsize=14)
plt.tight_layout(); plt.show()

## 7. 16 张未标注图像推理

In [ ]:
labeled_stems_all = {s['stem'] for s in dataset.samples}
unlabeled_pngs = sorted([p for p in DEF_DIR.glob('*.png') if p.stem not in labeled_stems_all])
print(f'Unlabeled: {len(unlabeled_pngs)}')

def load_and_infer(png_path):
    img = np.asarray(PILImage.open(png_path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), PILImage.BILINEAR), np.float32)/255.
    t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(DEVICE)
    return img, infer(dino_trainer, t), infer(eupe_trainer, t)

results = [(pn.stem, *load_and_infer(pn)) for pn in unlabeled_pngs]

for page, start in enumerate([0, 8]):
    batch = results[start:start+8]; n = len(batch)
    fig, axes = plt.subplots(n, 3, figsize=(15, 3.5*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for row, (stem, img, dp, ep) in enumerate(batch):
        axes[row, 0].imshow(img, cmap='gray'); axes[row, 0].set_title(stem, fontsize=8); axes[row, 0].axis('off')
        for col, (mask, name) in enumerate([(dp, f'DINOv3 d={int((dp==1).sum())} b={int((dp==2).sum())}'),
                                             (ep, f'EUPE d={int((ep==1).sum())} b={int((ep==2).sum())}')]):
            axes[row, col+1].imshow(img, cmap='gray')
            axes[row, col+1].imshow(mask_to_rgb(mask), alpha=0.5)
            axes[row, col+1].set_title(name, fontsize=8); axes[row, col+1].axis('off')
    fig.suptitle(f'Unlabeled Defect Seg — Page {page+1}/2', fontsize=13)
    plt.tight_layout(); plt.show()

print(f"\n{'stem':12s}  {'DINO dark':>10s} {'DINO bright':>10s} {'EUPE dark':>10s} {'EUPE bright':>10s}")
print('-'*60)
for stem, _, dp, ep in results:
    print(f'{stem:12s}  {int((dp==1).sum()):10d} {int((dp==2).sum()):10d} {int((ep==1).sum()):10d} {int((ep==2).sum()):10d}')

## 8. 自定义挑选可视化

修改下面 `PICK_STEMS` 列表选择要查看的图像名。

In [ ]:
# Custom pick visualization - edit PICK_STEMS to choose images
# Right-click figure -> Save Image As, or set SAVE_FIG=True
SAVE_FIG = False
PICK_STEMS = ['FeTe_0005', 'FeTe_0006', 'FeTe_0008', 'FeTe_0009', 'FeTe_0014', 'FeTe_0017']  # ← 改这里

# Load best heads
dino_ckpt = torch.load(DINO_BEST, map_location=DEVICE, weights_only=True)
dino_trainer.head.load_state_dict(dino_ckpt.get("head_state_dict", dino_ckpt.get("head", dino_ckpt))); dino_trainer.eval()
eupe_ckpt = torch.load(EUPE_BEST, map_location=DEVICE, weights_only=True)
eupe_trainer.head.load_state_dict(eupe_ckpt.get("head_state_dict", eupe_ckpt.get("head", eupe_ckpt))); eupe_trainer.eval()

# Build lookup: all PNGs in data dir
all_pngs = {p.stem: p for p in DEF_DIR.glob("*.png")}

@torch.no_grad()
def load_and_predict(path):
    img = np.asarray(PILImage.open(path).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), PILImage.BILINEAR), dtype=np.float32) / 255.
    t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(DEVICE)
    d_pred = dino_trainer(t).argmax(dim=1)[0].cpu().numpy()
    e_pred = eupe_trainer(t).argmax(dim=1)[0].cpu().numpy()
    return img, d_pred, e_pred

# Filter to stems that actually exist on disk
valid = []
for stem in PICK_STEMS:
    p = all_pngs.get(stem)
    if p is not None:
        valid.append((stem, p))
    else:
        print(f"Warning: {stem} not found, skipping")

n = len(valid)
if n == 0:
    print("No valid files found for PICK_STEMS")
else:
    fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
    if n == 1: axes = axes.reshape(1, -1)
    for row, (stem, png_path) in enumerate(valid):
        has_label = stem in {s["stem"] for s in dataset.samples}
        tag = " [labeled]" if has_label else ""
        img, d_pred, e_pred = load_and_predict(png_path)
        # Col 0: input
        axes[row, 0].imshow(img, cmap="gray")
        axes[row, 0].set_title(f"{stem}{tag}", fontsize=10)
        axes[row, 0].axis("off")
        # Col 1: DINOv3 overlay
        axes[row, 1].imshow(img, cmap="gray")
        d_ov = np.zeros((*d_pred.shape, 4))
        d_ov[d_pred == 1] = [1,0,0,0.35]  # red=dark
        d_ov[d_pred == 2] = [1,1,0,0.35]  # yellow=bright
        axes[row, 1].imshow(d_ov)
        axes[row, 1].set_title(f"DINOv3", fontsize=10)
        axes[row, 1].axis("off")
        # Col 2: EUPE overlay
        axes[row, 2].imshow(img, cmap="gray")
        e_ov = np.zeros((*e_pred.shape, 4))
        e_ov[e_pred == 1] = [1,0,0,0.35]  # red=dark
        e_ov[e_pred == 2] = [1,1,0,0.35]  # yellow=bright
        axes[row, 2].imshow(e_ov)
        axes[row, 2].set_title(f"EUPE", fontsize=10)
        axes[row, 2].axis("off")
    plt.tight_layout(pad=1.0)
    if SAVE_FIG:
        save_dir = REPO_ROOT / "artifacts" / "custom_pick"
        save_dir.mkdir(parents=True, exist_ok=True)
        fname = save_dir / f"pick_{PICK_STEMS[0]}.png"
        fig.savefig(str(fname), dpi=150, bbox_inches="tight", facecolor="white")
        print(f"Saved: {fname}")
    plt.show()
    # Stats
    print()
    hdr = "{:12s}  {:>10s}  {:>10s}  {:>10s}  {:>10s}".format(
        "stem", "DINO cls1", "DINO cls2", "EUPE cls1", "EUPE cls2")
    print(hdr)
    print("-" * 65)
    for stem, png_path in valid:
        img, dp, ep = load_and_predict(png_path)
        print("{s:12s}  {m1:10d}  {m2:10d}  {e1:10d}  {e2:10d}".format(
            s=stem, m1=int((dp==1).sum()), m2=int((dp==2).sum()),
            e1=int((ep==1).sum()), e2=int((ep==2).sum())))
